# PureReason — RL Reasoning Training (No KL Divergence)

Trains Qwen2.5-1.5B on math, code, logic, Lean proofs, Sokoban & spreadsheet tasks using GRPO without KL penalty.

**Requirements:** T4 GPU (free Colab tier)  
**Runtime:** ~2-4 hours for 5000 steps

In [ ]:
#@title 1. Clone repo & install Python dependencies (~2 min)
import os, sys

REPO_URL = "https://github.com/YOUR_USERNAME/purereason.git"  #@param {type:"string"}
BRANCH = "master"  #@param {type:"string"}

if not os.path.exists("purereason"):
    !git clone --branch {BRANCH} {REPO_URL}
%cd purereason

!pip install -q torch transformers trl peft bitsandbytes datasets accelerate sympy wandb
print("Python deps installed.")

In [ ]:
#@title 2. Install Lean 4 + Mathlib (~15-30 min, one-time)
import os

# Install elan
if not os.path.exists(os.path.expanduser("~/.elan/bin/lean")):
    !curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y

os.environ["PATH"] = os.path.expanduser("~/.elan/bin") + ":" + os.environ["PATH"]
!elan default leanprover/lean4:stable
!lean --version

# Create lake project with mathlib (one-time)
if not os.path.exists("/tmp/lean_verify/.lake"):
    !mkdir -p /tmp/lean_verify
    !cat > /tmp/lean_verify/lakefile.lean << 'LAKEEOF'
import Lake
open Lake DSL
require mathlib from git "https://github.com/leanprover-community/mathlib4"
package lean_verify where moreLeanArgs := #["-DwarningAsError=false"]
@[default_target] lean_lib LeanVerify where
LAKEEOF
    !cd /tmp/lean_verify && lake update && lake exe cache get
else:
    print("Mathlib cache already set up.")

In [ ]:
#@title 3. Start Training (~2-4 hours)
import os
os.environ["PATH"] = os.path.expanduser("~/.elan/bin") + ":" + os.environ["PATH"]

%cd /content/purereason
!python src/train.py --preset colab